## Step 1: Import Required Libraries

In [18]:
import mssql_python
import csv
import json
from tqdm import tqdm

## Step 2: Configure Connection

Set up the SQL Server connection string. Update this with your server details.

In [19]:
CONNECTION_STRING = "Server=localhost;Database=master;UID=sa;PWD={z+/3GyAx6zUerW0r};TrustServerCertificate=yes;"
csv_file = "finefoodembeddings.csv"
table_name = "FoodReviewEmbeddings"

## Step 3: Create a Data Iterator

The Bulk Copy API accepts an iterator that yields tuples of data. This approach:
- Streams data without loading everything into memory
- Allows for data transformation on-the-fly
- Provides progress tracking

Each row contains:
- `Id`: Unique identifier
- `ProductId`: Amazon product ID
- `Score`: Review rating (1-5)
- `Summary`: Review summary text
- `Text`: Full review text
- `Combined`: Concatenated text for embedding
- `Embedding`: 1536-dimensional vector (OpenAI embedding)

In [20]:
class EmbeddingIterator:
    """Iterator that reads embeddings from CSV and converts to f32 lists"""
    
    def __init__(self, csv_file, max_rows=None, progress_bar=None):
        self.csv_file = csv_file
        self.max_rows = max_rows
        self.progress_bar = progress_bar
        
    def __iter__(self):
        with open(self.csv_file, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for i, row in enumerate(reader):
                if self.max_rows is not None and i >= self.max_rows:
                    break
                
                # Parse ID
                row_id = int(row.get('Id', 0))
                
                # Get text fields
                product_id = row.get('ProductId', '')
                score = int(row.get('Score', 0))
                summary = row.get('Summary', '')
                text = row.get('Text', '')
                combined = row.get('combined', '')
                
                # Parse embedding from JSON string and convert to list of floats (f32)
                embedding_str = row.get('vector', '[]')
                embedding = json.loads(embedding_str)
                # Ensure all values are float32
                embedding_f32 = [float(val) for val in embedding]
                
                # Update progress bar if provided
                if self.progress_bar is not None:
                    self.progress_bar.update(1)
                
                # Yield tuple of (id, product_id, score, summary, text, combined, embedding_vector)
                yield (row_id, product_id, score, summary, text, combined, embedding_f32)

## Step 4: Create Target Table

Create a SQL Server table with a VECTOR column to store embeddings. The table schema includes:
- Standard data columns (Id, ProductId, Score, etc.)
- A `VECTOR(1536)` column for storing embeddings

In [21]:
print(f"Connecting to SQL Server...")
conn = mssql_python.connect(CONNECTION_STRING)
cursor = conn.cursor()

print(f"Creating table '{table_name}'...")

# Drop table if exists
drop_table_sql = f"""
IF OBJECT_ID('{table_name}', 'U') IS NOT NULL
    DROP TABLE {table_name};
"""
cursor.execute(drop_table_sql)

# Create table with vector support
create_table_sql = f"""
CREATE TABLE {table_name} (
    Id INT PRIMARY KEY,
    ProductId NVARCHAR(50),
    Score INT,
    Summary NVARCHAR(500),
    ReviewText NVARCHAR(MAX),
    CombinedText NVARCHAR(MAX),
    Embedding VECTOR(1536)
);
"""
cursor.execute(create_table_sql)
conn.commit()

print(f"✓ Table '{table_name}' created successfully!\n")
print(f"Table schema:")
print(f"  - Id: INT PRIMARY KEY")
print(f"  - ProductId: NVARCHAR(50)")
print(f"  - Score: INT")
print(f"  - Summary: NVARCHAR(500)")
print(f"  - ReviewText: NVARCHAR(MAX)")
print(f"  - CombinedText: NVARCHAR(MAX)")
print(f"  - Embedding: VECTOR(1536)")

Connecting to SQL Server...
Creating table 'FoodReviewEmbeddings'...
✓ Table 'FoodReviewEmbeddings' created successfully!

Table schema:
  - Id: INT PRIMARY KEY
  - ProductId: NVARCHAR(50)
  - Score: INT
  - Summary: NVARCHAR(500)
  - ReviewText: NVARCHAR(MAX)
  - CombinedText: NVARCHAR(MAX)
  - Embedding: VECTOR(1536)


## Step 5: Perform Bulk Copy

Now we'll use the `bulk_copy()` method to efficiently load the data. Key benefits:

- **Performance**: Much faster than individual INSERT statements
- **Memory Efficient**: Streams data without loading entire dataset into memory
- **Progress Tracking**: Real-time feedback on copy progress
- **Type Safety**: Automatic type conversion and validation

In [22]:
# Count total rows for progress bar
total_rows = sum(1 for _ in open(csv_file, 'r', encoding='utf-8')) - 1  # Subtract header

print(f"\nStarting bulk copy of {total_rows:,} rows...\n")

# Create progress bar with tqdm
with tqdm(total=total_rows, desc="Copying data", unit=" rows") as pbar:
    # Create iterator with progress tracking
    embedding_data = EmbeddingIterator(csv_file, progress_bar=pbar)
    
    try:
        result = cursor.bulk_copy(
            table_name,
            embedding_data
        )
        print("\n✓ Bulk copy completed!")
        print("Result:")
        print(json.dumps(result, indent=2))
    except Exception as e:
        print("\n✗ Bulk copy failed!")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {e}")
        import traceback
        traceback.print_exc()


Starting bulk copy of 10,000 rows...



Copying data: 100%|██████████| 10000/10000 [00:06<00:00, 1501.66 rows/s]


✓ Bulk copy completed!
Result:
{
  "rows_copied": 10000,
  "batch_count": 1,
  "elapsed_time": 6.489084839,
  "rows_per_second": 1541.0493541244946
}


## Step 6: Verify the Data

Let's query the table to verify the data was loaded correctly.

In [25]:
# Query to verify data
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
row_count = cursor.fetchone()[0]
print(f"Total rows in table: {row_count:,}")

# Get sample rows
cursor.execute(f"SELECT TOP 3 Id, ProductId, Score, Summary FROM {table_name}")
print("\nSample rows:")
for row in cursor.fetchall():
    print(f"  ID: {row[0]}, Product: {row[1]}, Score: {row[2]}, Summary: {row[3][:50]}...")

Total rows in table: 10,000

Sample rows:
  ID: 1, Product: B001E4KFG0, Score: 5, Summary: Good Quality Dog Food...
  ID: 2, Product: B00813GRG4, Score: 1, Summary: Not as Advertised...
  ID: 3, Product: B000LQOCH0, Score: 4, Summary: "Delight" says it all...


## Step 7: Cleanup

Close the database connection.

In [26]:
cursor.close()
conn.close()
print("Connection closed.")

Connection closed.


## Summary

This demo showed how to:
1. Create a custom iterator for streaming data
2. Handle complex data types (VECTOR columns)
3. Use the Bulk Copy API for high-performance data loading
4. Track progress during large data operations

### Performance Benefits

Compared to row-by-row INSERT statements, Bulk Copy provides:
- **10-100x faster** data loading
- **Minimal memory usage** through streaming
- **Built-in type conversion** and validation
- **Progress tracking** for long-running operations

### Common Use Cases

- ETL pipelines
- Data migration
- Machine learning feature stores
- Log aggregation
- Time-series data ingestion